# RandomForest Scaling In — Optimized Entry Timing

**Finding from notebook 6:** RandomForest beats LogisticRegression at 83.2% vs 79.3% per-candle accuracy.

**Finding from the elapsed analysis:** RF is weak early (57% at 0-10%) but dominates late (85%+ at 60%+). LR is more consistent across the candle.

**This notebook tests:** What's the best scaling strategy for RF specifically? Should we enter later than LR?

In [ ]:
import json
import random
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from technicals import compute_all
from tqdm import tqdm

random.seed(42)
np.random.seed(42)
MAX_BID = 0.85

## 1. Train both models + compute predictions

In [ ]:
rows = []
with open(Path("../data/legacy_features.jsonl")) as f:
    for line in f:
        rows.append(json.loads(line))
df = pd.DataFrame(rows)
df["target"] = (df["outcome"] == "UP").astype(int)
NON_FEAT = {
    "candle_id",
    "timestamp",
    "elapsed_pct",
    "btc_price",
    "up_best_bid",
    "up_best_ask",
    "up_bid_depth",
    "up_ask_depth",
    "down_best_bid",
    "down_best_ask",
    "down_bid_depth",
    "down_ask_depth",
    "market_volume",
    "outcome",
    "target",
}
feat_cols = [c for c in df.columns if c not in NON_FEAT]
df[feat_cols] = df[feat_cols].fillna(0.0)
scaler = StandardScaler()
X_train = scaler.fit_transform(df[feat_cols].values)
y_train = df["target"].values

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
rf = RandomForestClassifier(n_estimators=200, max_depth=15, min_samples_leaf=20, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
print(f"Trained on {df['candle_id'].nunique()} candles")

In [ ]:
conn = sqlite3.connect(str(Path("../data/collection.db")))
candles_db = pd.read_sql("SELECT * FROM candles ORDER BY start_time", conn)
snaps_db = pd.read_sql("SELECT * FROM snapshots ORDER BY candle_id, timestamp", conn)
conn.close()

new_candles = []
for _, cr in candles_db.iterrows():
    cid = cr["candle_id"]
    srows = snaps_db[snaps_db["candle_id"] == cid]
    snaps = []
    for _, s in srows.iterrows():
        ob = json.loads(s["orderbook_json"])
        snaps.append(
            {
                "timestamp": s["timestamp"],
                "tick_timestamp": s["tick_timestamp"],
                "elapsed_pct": s["elapsed_pct"],
                "btc_price": s["btc_price"],
                "btc_bid": s["btc_bid"],
                "btc_ask": s["btc_ask"],
                "up_bids": [ob["up_bids"][0]] if ob.get("up_bids") else [],
                "up_asks": [ob["up_asks"][0]] if ob.get("up_asks") else [],
                "down_bids": [ob["down_bids"][0]] if ob.get("down_bids") else [],
                "down_asks": [ob["down_asks"][0]] if ob.get("down_asks") else [],
                "up_last_trade": s["up_last_trade"],
                "down_last_trade": s["down_last_trade"],
                "market_volume": s["market_volume"],
            }
        )
    new_candles.append(
        {
            "candle_id": cid,
            "start_time": cr["start_time"],
            "end_time": cr["end_time"],
            "open": cr["open"],
            "high": cr["high"],
            "low": cr["low"],
            "close": cr["close"],
            "volume": cr["volume"],
            "outcome": cr["outcome"],
            "final_ret": cr["final_ret"],
            "snapshots": snaps,
        }
    )

# Compute per-snapshot predictions for both models
prior = []
all_cd = []
for candle in tqdm(new_candles, desc="Computing predictions"):
    truth = 1 if candle["outcome"] == "UP" else 0
    snaps = candle["snapshots"]
    if len(snaps) < 5:
        prior.append(candle)
        continue
    sd = []
    for si in range(len(snaps)):
        snap = snaps[si]
        indicators = compute_all(prior, candle["open"], snaps[: si + 1])
        row_data = {f: indicators.get(f, 0.0) if indicators.get(f) is not None else 0.0 for f in feat_cols}
        X = scaler.transform(pd.DataFrame([row_data])[feat_cols].values)
        lr_prob = lr.predict_proba(X)[0, 1]
        rf_prob = rf.predict_proba(X)[0, 1]
        sd.append(
            {
                "tick": si,
                "elapsed_pct": snap["elapsed_pct"],
                "lr_pred": 1 if lr_prob >= 0.5 else 0,
                "lr_prob": lr_prob,
                "rf_pred": 1 if rf_prob >= 0.5 else 0,
                "rf_prob": rf_prob,
                "up_ask": snap["up_asks"][0][0] if snap["up_asks"] else None,
                "down_ask": snap["down_asks"][0][0] if snap["down_asks"] else None,
            }
        )
    all_cd.append({"candle_id": candle["candle_id"], "truth": truth, "snapshots": sd})
    prior.append(candle)

print(f"Computed predictions for {len(all_cd)} candles")

## 2. Scaling-in engine (supports both models)

In [ ]:
def run_scaling(name, entry_points, model_key="lr", bet=10.0):
    """model_key: 'lr' or 'rf' — which model's predictions to use."""
    pred_key = f"{model_key}_pred"
    bal = 1000.0
    hist = [bal]
    nb, wins, entered, skipped = 0, 0, 0, 0
    for cd in all_cd:
        sd = cd["snapshots"]
        truth = cd["truth"]
        entries = []
        first_dir = None
        for min_e, n_c in entry_points:
            for i in range(max(n_c - 1, 0), len(sd)):
                if sd[i]["elapsed_pct"] < min_e:
                    continue
                if any(i <= pt for pt, _, _ in entries):
                    continue
                if n_c > 1 and not all(sd[i - j][pred_key] == sd[i][pred_key] for j in range(n_c)):
                    continue
                d = sd[i][pred_key]
                if first_dir is None:
                    first_dir = d
                elif d != first_dir:
                    break
                ask = sd[i]["up_ask"] if d == 1 else sd[i]["down_ask"]
                if ask is None or not np.isfinite(ask) or ask <= 0 or ask >= MAX_BID:
                    continue
                entries.append((i, d, ask))
                break
        if not entries:
            skipped += 1
            continue
        entered += 1
        for _, d, ask in entries:
            if bal < bet:
                break
            nb += 1
            if d == truth:
                bal += (bet / ask) * (1.0 - ask)
                wins += 1
            else:
                bal -= bet
        hist.append(bal)
    wr = wins / nb if nb else 0
    peak = hist[0]
    mdd = 0
    for h in hist:
        if h > peak:
            peak = h
        dd = (peak - h) / peak
        if dd > mdd:
            mdd = dd
    return {
        "name": name,
        "balance": bal,
        "history": hist,
        "total_bets": nb,
        "entered": entered,
        "skipped": skipped,
        "wins": wins,
        "win_rate": wr,
        "return": (bal - 1000) / 1000 * 100,
        "max_dd": mdd,
    }

## 3. RF strategies — delayed entry

**Key insight:** RF is weak at 0-10% elapsed (57%) but strong at 30%+ (75%+). So RF should start later than LR.

In [ ]:
strategies = [
    ("LR 1x e5%", [(0.05, 3)], "lr"),
    ("LR 2x e5%+e50%", [(0.05, 3), (0.50, 3)], "lr"),
    ("RF 1x e5%", [(0.05, 3)], "rf"),
    ("RF 2x e5%+e50%", [(0.05, 3), (0.50, 3)], "rf"),
    ("RF 1x e20%", [(0.20, 3)], "rf"),
    ("RF 1x e30%", [(0.30, 3)], "rf"),
    ("RF 1x e40%", [(0.40, 3)], "rf"),
    ("RF 2x e20%+e50%", [(0.20, 3), (0.50, 3)], "rf"),
    ("RF 2x e30%+e60%", [(0.30, 3), (0.60, 3)], "rf"),
    ("RF 2x e30%+e55%", [(0.30, 3), (0.55, 3)], "rf"),
    ("RF 2x e40%+e65%", [(0.40, 3), (0.65, 3)], "rf"),
    ("RF 3x e20%+e40%+e60%", [(0.20, 3), (0.40, 3), (0.60, 3)], "rf"),
    ("RF 3x e30%+e50%+e70%", [(0.30, 3), (0.50, 3), (0.70, 3)], "rf"),
]

results = []
print(f"{'Strategy':<30} {'Bets':>5} {'WR':>6} {'Balance':>10} {'Return':>8} {'MaxDD':>7}")
print("-" * 70)
for name, eps, model_key in strategies[:-1]:
    r = run_scaling(name, eps, model_key)
    results.append(r)
    print(
        f"{r['name']:<30} {r['total_bets']:>5} {r['win_rate'] * 100:>5.1f}% ${r['balance']:>9.2f} {r['return']:>+7.1f}% {r['max_dd'] * 100:>6.1f}%"
    )

# Hybrid: LR for first entry, RF for second
bal = 1000.0
hist = [bal]
nb = 0
wins = 0
for cd in all_cd:
    sd = cd["snapshots"]
    truth = cd["truth"]
    entries = []
    # Entry 1: LR at e>=5%
    for i in range(2, len(sd)):
        if sd[i]["elapsed_pct"] < 0.05:
            continue
        if sd[i]["lr_pred"] == sd[i - 1]["lr_pred"] == sd[i - 2]["lr_pred"]:
            d = sd[i]["lr_pred"]
            ask = sd[i]["up_ask"] if d == 1 else sd[i]["down_ask"]
            if ask and np.isfinite(ask) and 0 < ask < MAX_BID:
                entries.append((d, ask))
            break
    # Entry 2: RF at e>=50%, must agree with entry 1
    if entries:
        first_dir = entries[0][0]
        for i in range(2, len(sd)):
            if sd[i]["elapsed_pct"] < 0.50:
                continue
            if sd[i]["rf_pred"] == sd[i - 1]["rf_pred"] == sd[i - 2]["rf_pred"] == first_dir:
                ask = sd[i]["up_ask"] if first_dir == 1 else sd[i]["down_ask"]
                if ask and np.isfinite(ask) and 0 < ask < MAX_BID:
                    entries.append((first_dir, ask))
                break
    for d, ask in entries:
        if bal < 10:
            break
        nb += 1
        if d == truth:
            bal += (10.0 / ask) * (1.0 - ask)
            wins += 1
        else:
            bal -= 10.0
    hist.append(bal)

wr = wins / nb if nb else 0
peak = hist[0]
mdd = 0
for h in hist:
    if h > peak:
        peak = h
    dd = (peak - h) / peak
    if dd > mdd:
        mdd = dd
hybrid = {
    "name": "Hybrid: LR@5%+RF@50%",
    "balance": bal,
    "history": hist,
    "total_bets": nb,
    "wins": wins,
    "win_rate": wr,
    "return": (bal - 1000) / 1000 * 100,
    "max_dd": mdd,
}
results.append(hybrid)
print(
    f"{hybrid['name']:<30} {nb:>5} {wr * 100:>5.1f}% ${bal:>9.2f} {(bal - 1000) / 1000 * 100:>+7.1f}% {mdd * 100:>6.1f}%"
)

## 4. Equity curves

In [ ]:
# Top performers
top = sorted(results, key=lambda r: -r["balance"])[:6]

fig, ax = plt.subplots(figsize=(16, 7))
colors = ["#2ecc71", "#3498db", "#e74c3c", "#f39c12", "#9b59b6", "#1abc9c"]
for r, color in zip(top, colors, strict=True):
    ax.plot(
        r["history"],
        label=f"{r['name']} -> ${r['balance']:,.0f} ({r['return']:+.0f}%, WR={r['win_rate'] * 100:.0f}%)",
        color=color,
        linewidth=1.5,
    )
ax.axhline(1000, color="gray", linestyle="--", alpha=0.3)
ax.set_xlabel("Candle #")
ax.set_ylabel("Balance ($)")
ax.set_title("Top 6 Strategies — LR vs RF vs Hybrid")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Conclusion

**Key findings:**

**RF accuracy by elapsed %:**

| Elapsed | LR Accuracy | RF Accuracy | Winner |
|---------|------------|------------|--------|
| 0-10% | 56.8% | 55.9% | Tie |
| 10-20% | 58.1% | 57.6% | Tie |
| 20-30% | 59.9% | 63.8% | **RF** |
| 30-40% | 66.3% | 74.6% | **RF** |
| 40-50% | 72.1% | 79.4% | **RF** |
| 50-60% | 72.9% | 80.1% | **RF** |
| 60-70% | 75.8% | 83.9% | **RF** |
| 70-80% | 79.9% | 85.4% | **RF** |

RF is clearly better after 20% elapsed, but useless in the first 20% (exactly as bad as LR). This means:

1. **RF should not enter early** — the 3-consecutive trigger at 5% gives RF bad entries
2. **RF shines as a confirmation model** — use it for the 2nd/3rd entry at 40%+
3. **The hybrid approach (LR early + RF late) should combine the best of both**

*(Run the notebook to see which strategy wins.)*